In [1]:
from PIL import Image

from ultralytics import YOLO

from pathlib import Path

import sys

# 프로젝트 루트 추가
ROOT = Path().resolve().parent   # fastapi-agent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 클래스만 import
from config.settings import Settings

# 인스턴스 생성 (env 자동 로드)
settings = Settings()

# yolo = YOLO("yolov8n.pt")
yolo = YOLO("yolov8n-oiv7.pt")


In [ ]:
def detect_and_crop(image_path):
    img = Image.open(image_path).convert("RGB")

    r = yolo(image_path, conf=0.1)[0]

    if len(r.boxes) == 0:
        return None

    boxes = r.boxes.xyxy
    confs = r.boxes.conf
    clss = r.boxes.cls

    idx = confs.argmax().item()

    x1, y1, x2, y2 = map(int, boxes[idx])


    # bounding box 크기
    w = x2 - x1
    h = y2 - y1

    img_x, img_y = img.size
    bbox_area = w * h
    img_area = img_x * img_y

    area_ratio = bbox_area / img_area


    min_area = 10

    if w < min_area or h < min_area:
        print(f"최소 크기 미만 : {img} | {min_area}")
        return None

    min_ratio = 0.01

    if area_ratio < min_ratio:
        print(f"최소 비율 미만 : {img} | {area_ratio}")
        return None


    crop = img.crop((x1, y1, x2, y2))

    # cw, ch = crop.size

    # if min(cw, ch) < 64:
    #     crop = img

    return {
        "crop": crop,
        "confidence": float(confs[idx]),
        "class_id": int(clss[idx]),
        "class_name": yolo.names[int(clss[idx])],
        "bbox": [x1, y1, x2, y2],
        "area_ratio": float(area_ratio)
    }

In [3]:
def to_static_image_url(image_path: Path) -> str:
    STORAGE_DIR = IMAGE_DIR = ROOT / "storage"
    relative = image_path.relative_to(STORAGE_DIR)
    return f"/static/{relative.as_posix()}"

In [4]:
from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer('clip-ViT-B-32')
embedding_model = SentenceTransformer('clip-ViT-L-14')

c:\xampp\htdocs\www\projects\fastapi-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import uuid
from qdrant_client.http.models import PointStruct

IMAGE_DIR = ROOT / "storage" / "images" / "fruits"

image_paths = [
    p for p in IMAGE_DIR.glob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

print('=====')
print(len(image_paths))
print("=====")

points = []

for idx, image_path in enumerate(image_paths):
    result = detect_and_crop(image_path)

    if result is None:
        print(f"{idx} 오류 : {image_path}\n")
        continue
    else:
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding_model.encode(result['crop']).tolist(),
                payload={
                    "image_path": to_static_image_url(image_path),
                    "bbox": result['bbox'],
                    "area_ratio": result['area_ratio'],
                }
            )
        )


        print(f"{idx} {image_path} {result}")
        # print(f"{image_path} {to_static_image_url(image_path)}")

# print(points)

=====
17
=====

image 1/1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\03f9a5d8576b38fb610ea9ebd57ad2ba.jpg: 640x640 1 Apple, 364.5ms
Speed: 11.3ms preprocess, 364.5ms inference, 10.5ms postprocess per image at shape (1, 3, 640, 640)
0 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\03f9a5d8576b38fb610ea9ebd57ad2ba.jpg {'crop': <PIL.Image.Image image mode=RGB size=989x1012 at 0x27BFA4C9C70>, 'confidence': 0.30072808265686035, 'class_id': 10, 'class_name': 'Apple', 'bbox': [115, 83, 1104, 1095], 'area_ratio': 0.6950472222222223}

image 1/1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\0a734cbe4efc3caf56af9f8393ab4565.jpg: 640x640 1 Banana, 440.3ms
Speed: 31.1ms preprocess, 440.3ms inference, 7.0ms postprocess per image at shape (1, 3, 640, 640)
1 C:\xampp\htdocs\www\projects\fastapi-agent\storage\images\fruits\0a734cbe4efc3caf56af9f8393ab4565.jpg {'crop': <PIL.Image.Image image mode=RGB size=369x368 at 0x27BFA4F6150>, 'confidence':

In [6]:
from qdrant_client import AsyncQdrantClient
qdrant = AsyncQdrantClient()

await qdrant.upsert(
    collection_name="fruits",
    points=points,
)

print("DONE")

DONE
